# Tox21 Parquet Merge

This notebook vertically merges existing normalized Tox21 parquet files into one parquet file.

It is streaming and row-group based, so it does not load all Tox21 data into memory at once.

Expected source layout:

```text
Prediction/Tox21/BatchE001.parquet
Prediction/Tox21/BatchE002.parquet
...
Prediction/Tox21/BatchG004.parquet
```

Expected output:

```text
Prediction/Tox21/Tox21_All.parquet
```

In [1]:
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# Source directory containing BatchE/BatchG Tox21 parquet files.
TOX21_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21")

# Merged output. This file is intentionally excluded from future scans.
OUTPUT_PATH = TOX21_DIR / "Tox21_All.parquet"

# Set to True only when you intentionally want to replace an existing output file.
OVERWRITE_OUTPUT = False

TOX21_COLUMNS = [
    "SMILES",
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
]

TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("NR-AR", pa.float64()),
    pa.field("NR-AR-LBD", pa.float64()),
    pa.field("NR-AhR", pa.float64()),
    pa.field("NR-Aromatase", pa.float64()),
    pa.field("NR-ER", pa.float64()),
    pa.field("NR-ER-LBD", pa.float64()),
    pa.field("NR-PPAR-gamma", pa.float64()),
    pa.field("SR-ARE", pa.float64()),
    pa.field("SR-ATAD5", pa.float64()),
    pa.field("SR-HSE", pa.float64()),
    pa.field("SR-MMP", pa.float64()),
    pa.field("SR-p53", pa.float64()),
])

print(f"Tox21 directory: {TOX21_DIR}")
print(f"Output path:     {OUTPUT_PATH}")

Tox21 directory: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21
Output path:     C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All.parquet


## Scan Inputs

This cell finds normalized batch parquet files in `TOX21_DIR`, excluding merged output and temporary output files.

In [2]:
def discover_input_files(tox21_dir: Path, output_path: Path) -> list[Path]:
    files = []
    for path in sorted(tox21_dir.glob("*.parquet")):
        if path.resolve() == output_path.resolve():
            continue
        if path.name.endswith(".tmp.parquet"):
            continue
        if path.stem.startswith("Tox21_All"):
            continue
        files.append(path)
    return files


input_files = discover_input_files(TOX21_DIR, OUTPUT_PATH)
if not input_files:
    raise FileNotFoundError(f"No input parquet files found in {TOX21_DIR}")

total_rows = 0
total_bytes = 0
total_row_groups = 0
for path in input_files:
    metadata = pq.ParquetFile(path).metadata
    total_rows += metadata.num_rows
    total_row_groups += metadata.num_row_groups
    total_bytes += path.stat().st_size
    print(
        f"{path.name:20s} rows={metadata.num_rows:>12,} "
        f"row_groups={metadata.num_row_groups:>5,} "
        f"size={path.stat().st_size / 1024**2:>9.2f} MB"
    )

print("-" * 80)
print(f"Input files:      {len(input_files)}")
print(f"Total rows:       {total_rows:,}")
print(f"Total row groups: {total_row_groups:,}")
print(f"Total size:       {total_bytes / 1024**3:.2f} GiB")

BatchE001.parquet    rows=   1,808,525 row_groups=    2 size=   204.03 MB
BatchE002.parquet    rows=   5,025,728 row_groups=    5 size=   578.99 MB
BatchE003.parquet    rows=     499,997 row_groups=    1 size=    53.18 MB
BatchE004.parquet    rows=     499,996 row_groups=    1 size=    53.23 MB
BatchE005.parquet    rows=     499,996 row_groups=    1 size=    53.01 MB
BatchE006.parquet    rows=      63,172 row_groups=    1 size=     7.68 MB
BatchE007.parquet    rows=     619,894 row_groups=    1 size=    64.12 MB
BatchE008.parquet    rows=     309,224 row_groups=    1 size=    34.23 MB
BatchE009.parquet    rows=     240,691 row_groups=    1 size=    25.97 MB
BatchE010.parquet    rows=     719,995 row_groups=    1 size=    72.71 MB
BatchE011.parquet    rows=     719,988 row_groups=    1 size=    73.26 MB
BatchE012.parquet    rows=     329,219 row_groups=    1 size=    35.57 MB
BatchE013.parquet    rows=      73,103 row_groups=    1 size=     8.78 MB
BatchE014.parquet    rows=     699,993

## Validate Schemas

The existing files may use either Arrow `string` or `large_string` for `SMILES`. The merge normalizes `SMILES` to `large_string`.

In [3]:
def validate_input_schema(path: Path) -> None:
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in TOX21_COLUMNS if column not in schema.names]
    extra = [name for name in schema.names if name not in TOX21_COLUMNS]
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")
    if extra:
        raise ValueError(f"{path.name} has unexpected columns: {extra}")


for path in input_files:
    validate_input_schema(path)

print(f"Validated {len(input_files)} parquet file(s).")
print(TARGET_SCHEMA)

Validated 28 parquet file(s).
SMILES: large_string
NR-AR: double
NR-AR-LBD: double
NR-AhR: double
NR-Aromatase: double
NR-ER: double
NR-ER-LBD: double
NR-PPAR-gamma: double
SR-ARE: double
SR-ATAD5: double
SR-HSE: double
SR-MMP: double
SR-p53: double


## Merge Function

This function reads each source parquet row group, casts columns to a single target schema, and appends it to the merged output.

In [4]:
def align_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def merge_tox21_parquets(input_paths: list[Path], output_path: Path, overwrite: bool = False) -> dict:
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {output_path}. Set OVERWRITE_OUTPUT = True to replace it.")

    temp_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_path.exists():
        if overwrite:
            temp_path.unlink()
        else:
            raise FileExistsError(f"Temporary output already exists: {temp_path}")

    writer = None
    rows_written = 0
    row_groups_written = 0

    try:
        writer = pq.ParquetWriter(
            temp_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES"],
        )

        file_iter = tqdm(input_paths, desc="Merge Tox21 files", unit="file", dynamic_ncols=True)
        for path in file_iter:
            parquet_file = pq.ParquetFile(path)
            file_iter.set_postfix(file=path.name, rows=f"{parquet_file.metadata.num_rows:,}")

            row_group_iter = tqdm(
                range(parquet_file.metadata.num_row_groups),
                desc=f"{path.stem} row groups",
                unit="row group",
                leave=False,
                dynamic_ncols=True,
            )
            for row_group_index in row_group_iter:
                table = parquet_file.read_row_group(row_group_index, columns=TOX21_COLUMNS)
                table = align_table(table)
                writer.write_table(table)
                rows_written += table.num_rows
                row_groups_written += 1
                row_group_iter.set_postfix(total_rows=f"{rows_written:,}")

    finally:
        if writer is not None:
            writer.close()

    if rows_written == 0:
        temp_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; merge aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_path.replace(output_path)

    return {
        "output_path": str(output_path),
        "rows_written": rows_written,
        "row_groups_written": row_groups_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }

## Run Merge

Set `RUN_MERGE = True` when you are ready. The output will be `Tox21_All.parquet` in the Tox21 directory.

In [6]:
RUN_MERGE = True

if RUN_MERGE:
    result = merge_tox21_parquets(input_files, OUTPUT_PATH, overwrite=OVERWRITE_OUTPUT)
    print("Merge complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_MERGE = True to create the merged parquet file.")

Merge Tox21 files:   0%|          | 0/28 [00:00<?, ?file/s]

BatchE001 row groups:   0%|          | 0/2 [00:00<?, ?row group/s]

BatchE002 row groups:   0%|          | 0/5 [00:00<?, ?row group/s]

BatchE003 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchE004 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

...

BatchG001 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchG002 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchG003 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

BatchG004 row groups:   0%|          | 0/1 [00:00<?, ?row group/s]

Merge complete.
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All.parquet
rows_written: 15616177
row_groups_written: 33
size_gib: 1.5825024656951427


## Deduplicate Tox21 Parquet

This standalone cell can be run by itself. Set absolute input/output parquet paths, choose any columns for deduplication, and it writes a separate deduplicated parquet file.

The deduplication uses a temporary SQLite database on disk to keep memory low. It keeps the first row encountered for each configured deduplication key.

In [1]:
# Standalone cell: deduplicate a Tox21 parquet file.
# You can run this cell without running any previous cell in this notebook.

from pathlib import Path
import math
import sqlite3

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ?? Path settings: edit these absolute paths as needed ??
RUN_DEDUP = True

DEDUP_INPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All.parquet")
DEDUP_OUTPUT_PATH = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All_dedup.parquet")

# Pick any one or more columns from TOX21_COLUMNS.
# Typical Tox21 usage: ["SMILES"]
DEDUP_COLUMNS = ["SMILES"]

# Smaller batches use less memory. Increase only if your machine handles it comfortably.
DEDUP_BATCH_SIZE = 50_000

# Temporary SQLite key database. Default: next to the output parquet.
DEDUP_SQLITE_PATH = DEDUP_OUTPUT_PATH.with_name(DEDUP_OUTPUT_PATH.stem + "_seen.sqlite")

# Streaming dedup keeps the first row encountered for each dedup key.
DEDUP_KEEP = "first"

# Set to True only when you intentionally want to replace the dedup output file.
OVERWRITE_DEDUP_OUTPUT = False

# Remove the temporary SQLite key database after successful deduplication.
CLEAN_DEDUP_TEMP = True

TOX21_COLUMNS = [
    "SMILES",
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53",
]

TARGET_SCHEMA = pa.schema([
    pa.field("SMILES", pa.large_string()),
    pa.field("NR-AR", pa.float64()),
    pa.field("NR-AR-LBD", pa.float64()),
    pa.field("NR-AhR", pa.float64()),
    pa.field("NR-Aromatase", pa.float64()),
    pa.field("NR-ER", pa.float64()),
    pa.field("NR-ER-LBD", pa.float64()),
    pa.field("NR-PPAR-gamma", pa.float64()),
    pa.field("SR-ARE", pa.float64()),
    pa.field("SR-ATAD5", pa.float64()),
    pa.field("SR-HSE", pa.float64()),
    pa.field("SR-MMP", pa.float64()),
    pa.field("SR-p53", pa.float64()),
])


def validate_dedup_settings(columns: list[str], keep: str, batch_size: int) -> None:
    if keep != "first":
        raise ValueError("Only DEDUP_KEEP = 'first' is supported for streaming deduplication.")
    if not columns:
        raise ValueError("DEDUP_COLUMNS must contain at least one column.")
    missing = [column for column in columns if column not in TOX21_COLUMNS]
    if missing:
        raise ValueError(f"DEDUP_COLUMNS contains unknown columns: {missing}")
    if batch_size < 1:
        raise ValueError("DEDUP_BATCH_SIZE must be at least 1.")


def validate_input_schema(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Input parquet does not exist: {path}")
    schema = pq.ParquetFile(path).schema_arrow
    missing = [column for column in TOX21_COLUMNS if column not in schema.names]
    if missing:
        raise ValueError(f"Input parquet is missing required columns: {missing}")


def align_table(table: pa.Table) -> pa.Table:
    arrays = []
    for field in TARGET_SCHEMA:
        array = table[field.name]
        if array.type != field.type:
            array = pc.cast(array, field.type)
        arrays.append(array)
    return pa.Table.from_arrays(arrays, schema=TARGET_SCHEMA)


def _dedup_value(value):
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def _dedup_key(value_tuple: tuple) -> str:
    return repr(tuple(_dedup_value(value) for value in value_tuple))


def make_key_rows(table: pa.Table, dedup_columns: list[str]) -> list[tuple[int, str]]:
    key_columns = [table[column].to_pylist() for column in dedup_columns]
    return [(row_index, _dedup_key(values)) for row_index, values in enumerate(zip(*key_columns))]


def setup_seen_key_db(db_path: Path, overwrite: bool) -> sqlite3.Connection:
    if db_path.exists():
        if overwrite:
            db_path.unlink()
        else:
            raise FileExistsError(f"Temporary SQLite key DB already exists: {db_path}")

    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=OFF")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("PRAGMA temp_store=FILE")
    conn.execute("PRAGMA cache_size=-200000")
    conn.execute("CREATE TABLE seen_keys (key TEXT PRIMARY KEY)")
    conn.execute("CREATE TEMP TABLE batch_keys (pos INTEGER NOT NULL, key TEXT NOT NULL)")
    conn.execute("CREATE INDEX batch_keys_key_pos_idx ON batch_keys(key, pos)")
    conn.commit()
    return conn


def find_new_positions(conn: sqlite3.Connection, key_rows: list[tuple[int, str]]) -> list[int]:
    conn.execute("DELETE FROM batch_keys")
    conn.executemany("INSERT INTO batch_keys(pos, key) VALUES (?, ?)", key_rows)

    keep_rows = conn.execute("""
        SELECT MIN(b.pos) AS pos, b.key
        FROM batch_keys b
        LEFT JOIN seen_keys s ON s.key = b.key
        WHERE s.key IS NULL
        GROUP BY b.key
    """).fetchall()

    if keep_rows:
        conn.executemany("INSERT OR IGNORE INTO seen_keys(key) VALUES (?)", [(key,) for _, key in keep_rows])
    conn.commit()
    return [pos for pos, _ in keep_rows]


def filter_positions(table: pa.Table, keep_positions: list[int]) -> pa.Table:
    if not keep_positions:
        return table.slice(0, 0)
    keep_positions.sort()
    return table.take(pa.array(keep_positions, type=pa.int64()))


def deduplicate_tox21_parquet_sqlite(
    input_path: Path,
    output_path: Path,
    dedup_columns: list[str],
    batch_size: int,
    sqlite_path: Path,
    overwrite: bool = False,
    clean_temp: bool = True,
) -> dict:
    validate_dedup_settings(dedup_columns, DEDUP_KEEP, batch_size)
    validate_input_schema(input_path)
    if input_path.resolve() == output_path.resolve():
        raise ValueError("DEDUP_OUTPUT_PATH must be different from DEDUP_INPUT_PATH.")
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"Dedup output already exists: {output_path}. Set OVERWRITE_DEDUP_OUTPUT = True to replace it.")

    temp_output_path = output_path.with_name(output_path.stem + ".tmp.parquet")
    if temp_output_path.exists():
        if overwrite:
            temp_output_path.unlink()
        else:
            raise FileExistsError(f"Temporary dedup output already exists: {temp_output_path}")

    conn = setup_seen_key_db(sqlite_path, overwrite=overwrite)
    parquet_file = pq.ParquetFile(input_path)
    writer = None
    rows_read = 0
    rows_written = 0
    duplicate_rows_skipped = 0
    batches_written = 0

    total_rows = parquet_file.metadata.num_rows
    progress = tqdm(
        parquet_file.iter_batches(batch_size=batch_size, columns=TOX21_COLUMNS),
        total=math.ceil(total_rows / batch_size),
        desc="Deduplicate Tox21 parquet",
        unit="batch",
        dynamic_ncols=True,
    )

    try:
        writer = pq.ParquetWriter(
            temp_output_path,
            TARGET_SCHEMA,
            compression="snappy",
            use_dictionary=["SMILES"],
        )

        for record_batch in progress:
            table = pa.Table.from_batches([record_batch])
            rows_read += table.num_rows
            table = align_table(table)

            key_rows = make_key_rows(table, dedup_columns)
            keep_positions = find_new_positions(conn, key_rows)
            duplicate_rows_skipped += table.num_rows - len(keep_positions)

            if keep_positions:
                filtered = filter_positions(table, keep_positions)
                writer.write_table(filtered)
                rows_written += filtered.num_rows
                batches_written += 1

            progress.set_postfix(
                read=f"{rows_read:,}",
                written=f"{rows_written:,}",
                skipped=f"{duplicate_rows_skipped:,}",
            )

    finally:
        progress.close()
        if writer is not None:
            writer.close()
        conn.close()

    if rows_written == 0:
        temp_output_path.unlink(missing_ok=True)
        raise RuntimeError("No rows were written; dedup aborted.")

    if output_path.exists() and overwrite:
        output_path.unlink()
    temp_output_path.replace(output_path)

    if clean_temp:
        sqlite_path.unlink(missing_ok=True)

    return {
        "input_path": str(input_path),
        "output_path": str(output_path),
        "dedup_columns": dedup_columns,
        "batch_size": batch_size,
        "sqlite_path": str(sqlite_path),
        "rows_read": rows_read,
        "rows_written": rows_written,
        "duplicate_rows_skipped": duplicate_rows_skipped,
        "batches_written": batches_written,
        "size_gib": output_path.stat().st_size / 1024**3,
    }


if RUN_DEDUP:
    result = deduplicate_tox21_parquet_sqlite(
        DEDUP_INPUT_PATH,
        DEDUP_OUTPUT_PATH,
        DEDUP_COLUMNS,
        DEDUP_BATCH_SIZE,
        DEDUP_SQLITE_PATH,
        overwrite=OVERWRITE_DEDUP_OUTPUT,
        clean_temp=CLEAN_DEDUP_TEMP,
    )
    print("Dedup complete.")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("Dry run only. Set RUN_DEDUP = True to create the deduplicated parquet file.")
    print(f"Dedup input:    {DEDUP_INPUT_PATH}")
    print(f"Dedup output:   {DEDUP_OUTPUT_PATH}")
    print(f"Dedup columns:  {DEDUP_COLUMNS}")
    print(f"Batch size:     {DEDUP_BATCH_SIZE:,}")
    print(f"SQLite DB:      {DEDUP_SQLITE_PATH}")


Deduplicate Tox21 parquet:   0%|          | 0/313 [00:00<?, ?batch/s]

Dedup complete.
input_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All.parquet
output_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All_dedup.parquet
dedup_columns: ['SMILES']
batch_size: 50000
sqlite_path: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\Tox21_All_dedup_seen.sqlite
rows_read: 15616177
rows_written: 10615944
duplicate_rows_skipped: 5000233
batches_written: 313
size_gib: 1.104172413237393


## Verify Output

Run this after the merge finishes.

In [ ]:
if OUTPUT_PATH.exists():
    merged = pq.ParquetFile(OUTPUT_PATH)
    print(f"Output:     {OUTPUT_PATH}")
    print(f"Rows:       {merged.metadata.num_rows:,}")
    print(f"Row groups: {merged.metadata.num_row_groups:,}")
    print(f"Size:       {OUTPUT_PATH.stat().st_size / 1024**3:.2f} GiB")
    print(merged.schema_arrow)
else:
    print(f"Output does not exist yet: {OUTPUT_PATH}")